In [ ]:
from pathlib import Path
import re
import matplotlib.pyplot as plt
import torch #modelin sinir ağı bu kütüphane üzerinden çalışacak
import cv2
import gradio as gr
from transformers import AutoProcessor, AutoModelForMultimodalLM
#Processor image i modelin anlayabileceği sayısal verilere(çok boyutlu sayısal diziler) çevirir
#AutoModelForMultimodalLM → MiniCPM-V modelini yükler config dosyasına göre modeli yükler
#AutoProcessor            → Görsel + metni modele hazırlar

In [ ]:
#modelin ve processorun ortama yüklenmesi 
MODEL_ID = "openbmb/MiniCPM-V-4.6-BNB"

processor = AutoProcessor.from_pretrained(MODEL_ID) #Sadece görsel ve metni ileride nasıl hazırlayacağını bilen processor nesnesini oluşturuyor.
model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        device_map="auto" #modelin nerede çalıştırılacağı otomatik belirlenecek (CPU veya GPU)
    )
model.eval() #eğitim değil de inference modunda çalıştırılacak. Katmanların çalışma davranışını inference'a uygun hale getirir.

In [ ]:
#image in yüklenmesi
IMAGE_PATH = Path("test_pictures\catt.jpg")

image_bgr = cv2.imread(str(IMAGE_PATH))
if image_bgr is None:
    raise ValueError(f"Image not found at {IMAGE_PATH}")
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB) #renk kanallarını değiştiriyoruz

In [ ]:
#chat modeli için inputu belirli bir konuşma yapısına getirilmesi
def build_messages(image_rgb, question):
    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_rgb
                },
                {
                    "type": "text",
                    "text": question
                }   
            ]
        }
    ]

In [ ]:
#modelin çalıştırılması
def ask_model(processor, model, image_rgb, question):
    #mesajın oluşturulması
    messages = build_messages(
        image_rgb,
        question
    )

    try:
        #mesajın processor a verilmesi ve sonucunda inputun artık modelin anlayacağı bir formata dönüştürülmesi
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True, #girdileri tokenlara çevir
            add_generation_prompt=True, #özel tokenleri ekle
            return_dict=True, #girdileri dictionary formatında döndür
            return_tensors="pt", #çıktının tensor formatında olmasını sağla
        )

        inputs = inputs.to(model.device)

        #modelin cevap üretmesi
        with torch.inference_mode(): #inference modunda olduğumuz için gradient hesaplarını kapatıyor
            outputs = model.generate(
                **inputs,
                max_new_tokens=100, #cevabın uzunluğu 100 token ile sınırlandırılıyor
            )

        generated_tokens = outputs[:, inputs["input_ids"].shape[1]:] #model çıktı üretirken çıkışın başında inputu da eklediği için onu kesip sadece modelin ürettiği kısmı alıyoruz
        #burada inputs["input_ids"] kısmı girdilerin dictionarysindeki girdi değerleirni alır
        #.shade[1] ile de tensorün boyutunu gösterir
        # : ile de tensör boyutu kadar olan kısımdaki tensörleri keser.

        #çıktı oluşturulurken girdinin hemen #modelin cevap üretmesi
        #arkasına yeni tokenler ekleniyor o yüzden cevabı oluşturuken promptu silmek gerekli

        # Modelin ürettiği cevabı token ID'lerinden string'e çevirme
        response = processor.batch_decode(
            generated_tokens,
            skip_special_tokens=True,  # <eos>, <pad>, <assistant> gibi özel tokenları gösterme
        )[0]  # Batch içindeki ilk cevabı al tek görsel tek soru olduğu için

        return response.strip()

    except Exception as error:
        print("Model error:", error)
        return ""

In [ ]:
#çıktıdan koordinatların çıkarılması
def parse_bbox(response):

    if not response:
        return None

    match = re.search(
        r"\[\s*"
        r"(\d+(?:\.\d+)?)\s*,\s*"
        r"(\d+(?:\.\d+)?)\s*,\s*"
        r"(\d+(?:\.\d+)?)\s*,\s*"
        r"(\d+(?:\.\d+)?)\s*\]",
        response
    )

    if match is None:
        return None

    return [
        float(value)
        for value in match.groups()
    ]

In [ ]:
#göreli konumu piksele çevirmek
def box_to_pixels(box, width, height):

        x1, y1, x2, y2 = box

        x1 = int(x1 * width)
        y1 = int(y1 * height)
        x2 = int(x2 * width)
        y2 = int(y2 * height)

        return x1, y1, x2, y2 

In [ ]:
#piksel koordinatalarına göre bbox çizimi
def draw_bbox(image, box, label="object"):

    if box is None:
        return image.copy()

    height, width = image.shape[:2]

    x1, y1, x2, y2 = box_to_pixels(
        box,
        width,
        height
    )

    # Koordinatlar geçerli mi?
    if x1 >= x2 or y1 >= y2:
        print("Invalid bounding box.")
        return image.copy()

    output_image = image.copy()

    cv2.rectangle(
        output_image,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

    cv2.putText(
        output_image,
        label,
        (x1, max(y1 - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    return output_image

In [ ]:
#modelin oluşturduğu resmi rgb ye çevirme ve yazdırma
def show_bbox_coordinates(box, image_bgr, label="object"):

    result_image = draw_bbox(
        image_bgr,
        box,
        label
    )

    result_image_rgb = cv2.cvtColor(
        result_image,
        cv2.COLOR_BGR2RGB
    )

    plt.imshow(result_image_rgb)
    plt.axis("off")
    plt.show()

In [ ]:
# %%
def process_image(image, question):

    if image is None:
        return None, "Please upload an image."

    if not question.strip():
        return image, "Please enter a question."

    response = ask_model(
        processor,
        model,
        image,
        question
    )

    box = parse_bbox(response)

    if box is None:
        return image, response

    result_image = draw_bbox(
        image,
        box,
        "object"
    )

    return result_image, response

In [ ]:
# %%
with gr.Blocks() as demo:

    gr.Markdown("# MiniCPM-V Visual Assistant")

    image_input = gr.Image(
        type="numpy",
        label="Upload Image"
    )

    question_input = gr.Textbox(
        label="Question",
        placeholder="Ask something about the image..."
    )

    ask_button = gr.Button("Ask")

    image_output = gr.Image(
        label="Result"
    )

    text_output = gr.Textbox(
        label="Model Response"
    )

    ask_button.click(
        fn=process_image,
        inputs=[
            image_input,
            question_input
        ],
        outputs=[
            image_output,
            text_output
        ]
    )

In [ ]:
demo.launch()